In [ ]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

# model construction

In [ ]:
using Symbolics,CairoMakie
numPointsX = collect(2:2)
logsOfHinverse = [1.0*i for i in 0:4]

cases=[]
#prefix="B"*string(tmpOrderBspace)*"_"*"w"*string(tmpWorderBspace)*"_"*string(tmpSupplementaryOrder)*"_"
prefix=""
L = 10.0*π
cases = push!(cases,(name=prefix*"λ_2",u=cos(x),β= 1))

@variables x
∂ = Differential(x)


misfit = Array{Float64,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))

modelFamily=[]

fig = Figure()
ax = Axis(fig[1, 1], xlabel="x", ylabel="model")
iH=5
for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    lines!(ax, X, model)
end
display(fig)
Nx=nothing
Δx=nothing
iH=nothing
for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    

    for iH ∈ eachindex(logsOfHinverse)
        modelName = name*string(Nx)
        ΔxTry = exp(-logsOfHinverse[iH])
        Nx = Int(L÷ΔxTry) +1
        Δx = L/(Nx-1)
        X = [Δx * (i-1) for i ∈ range(1,Nx)]
        models=[]
        model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
        models=push!(models, model)
        modelPoints = (Nx)
        tmpModel = (models=models, modelName=modelName, modelPoints=modelPoints)
        modelFamily=push!(modelFamily,tmpModel)
    end

   
end

for iPointsUsed ∈ eachindex(numPointsX), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,u,β = cases[iCase]
    
    models=modelFamily[iCase]
    q = mySimplify(β*∂(u))
    qₓ = mySimplify(∂(q))
    
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iPointsUsed)
    
    force = [Symbolics.value(substitute(qₓ,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    

    

    
end


# input parameters

In [ ]:

famousEquationType="1DpoissonHetero" #GT98
Δ = (1.0)

In [ ]:


# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1)



In [ ]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


In [ ]:
optRec=myProduceOrLoad(makeOPTsemiSymbolic,concreteParametersForOPTConstruction,"semiSymbolic")

In [ ]:
params=@strdict optRec=optRec modelFam=modelFamily[1] absorbingBoundaries=nothing maskedRegionInSpace=nothing

In [ ]:
numOpt=numericalOperatorConstruction(params)

In [ ]:
optRec["recette"].lhs.Ajiννᶜ

In [ ]:
Nt = 1
            
            
syntheticData=timeMarchingScheme(numOpt, Nt, Δnum,modelName;videoMode=false,sourceType="Explicit",sourceFull=sourceFull,iExperiment=iExperiment)
syntheticData=reduce(vcat,syntheticData)
analyticalData = [Symbolics.value(substitute(u,Dict(x=>X[i]))) for i ∈ range(1,Nx)]